# Migración: reestructura + renombrado + registro del volumen de fotos

Une `mover` y `registrar en la base de datos` en una sola operación, porque una vez que un archivo sale de `AUTORES/<nombre>/...` o `ESPECIE/<especie>/...`, esa pista se pierde para siempre si no se captura en el mismo momento del movimiento.

**Alcance de esta corrida:** solo las ramas `AUTORES`, `ESPECIE`, `EVENTOS`, `ESPACIOS`. **Se excluyen a propósito** `Para acomodar`, `__Para acomodar`, `FOTOS ARANZA` e `Ilustraciones Birds of the World`, porque siguen pendientes de una decisión de contenido tuya, no es un tema técnico. Cuando las resuelvas, se agregan en una corrida aparte.

**Convención de nombre:** `{RAMA}-{autor-o-SINAUTOR}-{especie-o-SINESP}-{hash6}.ext`. El hash sale de la ruta original completa, así que es reproducible: si el proceso se corta a la mitad, correrlo de nuevo no reprocesa lo ya movido ni genera nombres distintos para lo mismo.

**Flujo de dos pasos, no te lo saltes:**
1. Genera un **plan en seco** (`salida/Plan_Migracion.xlsx`) sin mover nada. Revísalo.
2. Solo cuando lo hayas revisado, cambias `EJECUTAR_MIGRACION_REAL = True` en la celda correspondiente y corres esa celda. Ahí sí mueve y registra de verdad.

In [ ]:
import os
import re
import shutil
import sqlite3
import hashlib
from pathlib import Path
from datetime import datetime

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter

RAIZ_GALERIA = Path(r"Y:\Galería")
DESTINO_UNIFICADO = RAIZ_GALERIA / "Fotos_Unificadas"
RUTA_BD = Path("bd") / "campuslab_volumen.db"
CARPETA_SALIDA = Path("salida")
CARPETA_SALIDA.mkdir(exist_ok=True)

RAMAS_INCLUIDAS = ("AUTORES", "ESPECIE", "EVENTOS", "ESPACIOS")

con = sqlite3.connect(RUTA_BD)

# Columna de control para poder pausar/reanudar sin reprocesar lo ya movido
cols_existentes = [r[1] for r in con.execute("PRAGMA table_info(staging_escaneo)")]
if "migrado" not in cols_existentes:
    con.execute("ALTER TABLE staging_escaneo ADD COLUMN migrado INTEGER DEFAULT 0")
    con.commit()

print(f"Destino unificado: {DESTINO_UNIFICADO}")
print(f"Ramas incluidas en esta corrida: {RAMAS_INCLUIDAS}")

In [ ]:
def slug(texto):
    if pd.isna(texto) or not texto:
        return None
    s = re.sub(r"[^\w\s-]", "", texto, flags=re.UNICODE)
    s = re.sub(r"\s+", "", s)
    return s or None

def hash_corto(ruta_original):
    return hashlib.sha1(ruta_original.encode("utf-8")).hexdigest()[:6]

def nombre_final(rama, autor, especie_cientifico, especie_comun, ruta_original, extension):
    autor_parte = slug(autor) or "SINAUTOR"
    especie_parte = slug(especie_cientifico) or slug(especie_comun) or "SINESP"
    h = hash_corto(ruta_original)
    return f"{rama}-{autor_parte}-{especie_parte}-{h}{extension}"

# Prueba rápida con casos ya conocidos, antes de usarla en serio
ejemplos = [
    ("AUTORES", "Gerardo Elizondo", None, None, r"Y:\Galería\FOTOS\AUTORES\GE Fotos Gerardo Elizondo\Insectos GE\bicho.jpg", ".jpg"),
    ("ESPECIE", None, "Amazilia violiceps", None, r"Y:\Galería\FOTOS\ESPECIE\AVES\Nombre Científico\Amazilia violiceps\colibri_01.jpg", ".jpg"),
    ("AUTORES", None, None, None, r"Y:\Galería\FOTOS\AUTORES\Fotos AGSaborio grandes\Revisadas\foto.jpg", ".jpg"),
]
for e in ejemplos:
    print(nombre_final(*e))

## Paso 1: plan en seco

No mueve nada todavía. Solo calcula a dónde iría cada archivo y con qué nombre nuevo.

In [ ]:
placeholders = ",".join("?" * len(RAMAS_INCLUIDAS))
df_staging = pd.read_sql(f'''
    SELECT * FROM staging_escaneo
    WHERE rama_raiz IN ({placeholders}) AND migrado = 0
''', con, params=RAMAS_INCLUIDAS)

print(f"Archivos pendientes de migrar en esta corrida: {len(df_staging):,}")

def calcular_destino(row):
    nombre = nombre_final(
        row["rama_raiz"], row["autor_sugerido"],
        row["especie_cientifico_sugerido"], row["especie_comun_sugerido"],
        row["ruta_completa"], row["extension"],
    )
    return nombre

if len(df_staging) == 0:
    df_staging["nombre_nuevo"] = pd.Series(dtype="object")
    df_staging["ruta_destino"] = pd.Series(dtype="object")
    print("No hay archivos pendientes, nada que planear.")
else:
    df_staging["nombre_nuevo"] = df_staging.apply(calcular_destino, axis=1)
    df_staging["ruta_destino"] = df_staging["nombre_nuevo"].apply(lambda n: str(DESTINO_UNIFICADO / n))

# Verificación de colisiones: no debería haber ninguna, el hash lo evita, pero lo confirmamos
n_colisiones = df_staging["nombre_nuevo"].duplicated().sum()
print(f"Colisiones de nombre detectadas: {n_colisiones}")
assert n_colisiones == 0, "Hay colisiones, revisar antes de continuar"

df_staging[["rama_raiz", "carpeta_contenedora", "nombre_archivo", "autor_sugerido", "confianza_autor",
            "especie_cientifico_sugerido", "especie_comun_sugerido", "confianza_especie",
            "nombre_nuevo", "ruta_completa", "ruta_destino"]].to_excel(
    CARPETA_SALIDA / "Plan_Migracion.xlsx", index=False
)
print(f"Plan guardado: {CARPETA_SALIDA / 'Plan_Migracion.xlsx'}")

## Paso 2: ejecución real

**No corras esta celda hasta haber revisado `Plan_Migracion.xlsx`.** Cambia `EJECUTAR_MIGRACION_REAL` a `True` solo cuando estés lista.

Por cada archivo: mueve el archivo físico, resuelve (o crea) el autor/especie en las tablas normalizadas, inserta el registro en `fotos`, y marca la fila de `staging_escaneo` como migrada. Si el proceso se corta a la mitad, correrlo de nuevo retoma donde se quedó, sin repetir ni duplicar.

In [ ]:
EJECUTAR_MIGRACION_REAL = False  # <-- cámbialo a True solo después de revisar el plan

def obtener_o_crear_autor(con, nombre, confianza):
    if pd.isna(nombre) or not nombre or confianza != "alta":
        return None
    cur = con.execute("SELECT id FROM autores WHERE nombre = ?", (nombre,))
    row = cur.fetchone()
    if row:
        return row[0]
    cur = con.execute("INSERT INTO autores (nombre) VALUES (?)", (nombre,))
    return cur.lastrowid

def obtener_o_crear_especie(con, cientifico, comun, confianza):
    cientifico = None if pd.isna(cientifico) else cientifico
    comun = None if pd.isna(comun) else comun
    if confianza != "alta" or not (cientifico or comun):
        return None
    cur = con.execute("SELECT id FROM especies WHERE nombre_cientifico IS ? AND nombre_comun IS ?", (cientifico, comun))
    row = cur.fetchone()
    if row:
        return row[0]
    cur = con.execute("INSERT INTO especies (nombre_cientifico, nombre_comun) VALUES (?, ?)", (cientifico, comun))
    return cur.lastrowid

if not EJECUTAR_MIGRACION_REAL:
    print("EJECUTAR_MIGRACION_REAL está en False. No se movió ni registró nada. Revisa el plan y vuelve aquí.")
else:
    DESTINO_UNIFICADO.mkdir(parents=True, exist_ok=True)
    log = []
    exitosos = fallidos = 0

    for _, row in df_staging.iterrows():
        origen = Path(row["ruta_completa"])
        destino = DESTINO_UNIFICADO / row["nombre_nuevo"]
        try:
            if not origen.exists():
                raise FileNotFoundError(f"No existe el origen: {origen}")
            shutil.move(str(origen), str(destino))

            autor_id = obtener_o_crear_autor(con, row["autor_sugerido"], row["confianza_autor"])
            especie_id = obtener_o_crear_especie(
                con, row["especie_cientifico_sugerido"], row["especie_comun_sugerido"], row["confianza_especie"]
            )
            estado = "usable" if (autor_id and especie_id) else "pendiente"

            con.execute('''
                INSERT INTO fotos (ruta_actual, ruta_original, autor_id, especie_id, estado, tamano_bytes, origen)
                VALUES (?, ?, ?, ?, ?, ?, 'volumen')
            ''', (str(destino), str(origen), autor_id, especie_id, estado, row["tamano_bytes"]))

            con.execute("UPDATE staging_escaneo SET migrado = 1 WHERE id = ?", (row["id"],))
            con.commit()
            log.append((str(origen), str(destino), "ok", ""))
            exitosos += 1
        except Exception as e:
            log.append((str(origen), str(destino), "error", str(e)))
            fallidos += 1

    df_log = pd.DataFrame(log, columns=["origen", "destino", "resultado", "detalle"])
    ruta_log = CARPETA_SALIDA / f"Log_Migracion_{datetime.now():%Y%m%d_%H%M%S}.csv"
    df_log.to_csv(ruta_log, index=False, encoding="utf-8")

    print(f"Exitosos: {exitosos:,}  Fallidos: {fallidos:,}")
    print(f"Log guardado: {ruta_log}")